# BERT: Pre-training of Deep Bidirectional Transformers - 실습 코드 1: BERT MLM 파인튜닝 (HuggingFace)

- Tutorial ID: `expand-bert-paper`
- Tutorial: BERT: Pre-training of Deep Bidirectional Transformers
- Section ID: `expand-bert-paper-code-1`
- Section: 실습 코드 1: BERT MLM 파인튜닝 (HuggingFace)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: BERT MLM 파인튜닝 (HuggingFace)
#
# 이 노트북은 "정답 코드를 한 번 실행해보는 것"이 아니라,
# BERT 논문의 핵심 아이디어인 Masked Language Model(MLM)이
# 실제로 어떤 텐서 연산으로 구현되는지 한 줄씩 따라가 보기 위한 실습 노트입니다.
#
# 학습 목표:
#   1) 토크나이저가 문장을 토큰/숫자로 바꾸는 과정과 [CLS]/[SEP]/[MASK] 같은
#      특수 토큰의 역할을 이해합니다.
#   2) 사전학습된 BERT가 [MASK] 자리에 올 단어를 예측하는 전체 과정
#      (logit -> softmax -> top-k)을 직접 따라가 봅니다.
#   3) 같은 [MASK] 자리라도 "앞뒤 문맥"에 따라 예측이 달라지는 모습을 보면서,
#      BERT가 왜 "양방향(bidirectional)" 모델인지 체감합니다.
#   4) BERT 논문의 마스킹 규칙(15%, 그중 80/10/10 비율)을 코드로 직접 확인하고,
#      작은 데이터셋으로 BERT를 실제로 한 번 더 학습(파인튜닝)시켜 봅니다.
#
# 읽는 순서:
#   1) Tokenizer 부분을 먼저 실행해서 문장이 어떤 토큰들로 쪼개지는지 확인하세요.
#   2) 모델을 불러온 뒤, 추론(inference) 코드를 한 줄씩 실행하며
#      shape이 어떻게 바뀌는지(logits -> probs -> top-5) print로 직접 확인하세요.
#   3) predict_mask() 함수를 여러 문장에 직접 호출해보며 예측이 어떻게
#      달라지는지 실험해보세요.
#   4) 파인튜닝 섹션에서는 작은 커피숍 말뭉치로 BERT를 다시 학습시키고,
#      학습 전/후 예측이 어떻게 달라지는지 비교해보세요.
#
# 주의:
#   - BERT는 GPT 계열과 달리 "미래 토큰을 가리는 인과적(causal) 구조"가
#     아닙니다. attention_mask는 [PAD]처럼 의미 없는 패딩 토큰을 무시하기
#     위한 장치일 뿐이며, 실제 문장 토큰들은 양방향으로 서로 모두 참조합니다.
#   - bert-base-uncased 가중치를 처음 받아올 때는 인터넷 연결과 약간의
#     다운로드 시간이 필요합니다 (약 440MB).
#   - GPU가 없어도 (속도만 느릴 뿐) 전체 코드가 정상적으로 동작합니다.
# ============================================================


## 들어가며: 이 노트북에서 무엇을 할까요?

BERT(Bidirectional Encoder Representations from Transformers)는 문장을 **양쪽 방향(왼쪽 + 오른쪽)의 문맥을 모두 보면서** 이해하는 Transformer 인코더 모델입니다. 이런 양방향 이해 능력을 학습시키기 위해 논문에서 사용한 핵심 학습 방법이 바로 **MLM(Masked Language Model, 마스크 언어 모델)**입니다.

> **MLM이 하는 일을 한 줄로 요약하면:** 문장에서 단어 몇 개를 가리고(`[MASK]`로 바꾸고), "원래 거기 있던 단어가 뭐였을까?"를 맞히도록 모델을 학습시킵니다.

이 노트북에서는 다음 순서로 직접 실습합니다.

1. **Tokenizer 살펴보기** — 문장이 숫자(토큰 id)로 어떻게 바뀌는지, `[CLS]`/`[SEP]`/`[MASK]` 같은 특수 토큰이 무엇인지 확인합니다.
2. **사전학습된 BERT로 빈칸 채우기 추론** — `bert-base-uncased`라는, 이미 대량의 텍스트로 학습이 끝난 모델을 불러와서 `[MASK]` 자리에 어떤 단어가 올지 예측해봅니다.
3. **양방향성(bidirectional) 직접 확인하기** — 같은 `[MASK]` 자리라도 문맥(특히 `[MASK]` *뒤쪽*의 단어들)이 바뀌면 예측이 어떻게 달라지는지 실험합니다.
4. **작은 데이터로 직접 파인튜닝(fine-tuning)** — '커피숍'을 주제로 한 짧은 문장 모음을 만들어서, BERT를 그 도메인에 맞게 한 번 더 학습시켜보고, 학습 전/후 예측이 달라지는 모습을 비교합니다.

> **참고:** 만약 Transformer의 Self-Attention 구조 자체가 아직 낯설다면, 이 시리즈의 앞쪽에 있는 Transformer 관련 실습 노트를 먼저 보고 오는 것을 추천합니다. 이 노트북에서는 "이미 만들어진 BERT를 어떻게 가져와서 쓰는가"에 집중합니다.


In [ ]:
# 처음 실행하는 경우, 또는 Google Colab에서 실행하는 경우 아래 줄의 주석(#)을 지우고 실행하세요.
# (이미 라이브러리가 설치되어 있다면 그대로 둬도 됩니다 - pip가 설치 여부만 빠르게 확인하고 지나갑니다)
# !pip install torch transformers datasets -q

from transformers import BertTokenizer, BertForMaskedLM
import torch

# -----------------------------------------------------------------
# 우리가 가져온 것들이 각각 무엇인지 짧게 정리하면:
#
#   - torch
#       파이토치(PyTorch). 텐서(다차원 배열) 연산과 자동미분(backpropagation)을
#       담당하는 딥러닝 프레임워크입니다. 모델 내부 연산은 모두 torch.Tensor로 이루어집니다.
#
#   - BertTokenizer
#       사람이 쓰는 문장을 BERT가 이해할 수 있는 "토큰 id 숫자열"로 바꿔주는 도구입니다.
#       (텍스트 -> 숫자, 숫자 -> 텍스트 양방향 변환을 모두 담당합니다)
#
#   - BertForMaskedLM
#       BERT 본체(Transformer Encoder) + "빈칸 채우기 전용 출력층(MLM head)"이
#       합쳐진 모델 클래스입니다. 뒤에서 자세히 다시 설명합니다.
#
# 참고: transformers/torch 라이브러리 버전에 따라 일부 경고(warning) 메시지의
# 문구가 다를 수 있지만, 이 노트북에서 다루는 핵심 동작 방식은 동일합니다.
# -----------------------------------------------------------------

print("torch 버전:", torch.__version__)


## 1. Tokenizer 살펴보기

BERT는 영어 "문장"을 그대로 입력받지 않습니다. 신경망은 결국 숫자 연산만 할 수 있기 때문에, 문장을 먼저 **토큰(token)** 단위로 자르고, 각 토큰을 **고유한 정수 id**로 바꿔주는 과정이 필요합니다. 이 역할을 하는 것이 **Tokenizer**입니다.

BERT는 **WordPiece**라는 방식의 토크나이저를 사용합니다. 동작 방식을 한마디로 설명하면:

> 자주 등장하는 단어는 통째로 하나의 토큰으로 쓰고, 드물게 등장하는 길거나 낯선 단어는 이미 알고 있는 더 작은 조각(서브워드, sub-word)들로 쪼개서 표현합니다.

예를 들어 `unbelievable` 같은 단어가 vocab(사전)에 통째로 없다면, `un` + `##believable` 처럼 둘 이상의 조각으로 나뉠 수 있습니다. 여기서 `##`은 "이 조각은 단어의 시작이 아니라 중간/끝부분이다"라는 표시입니다.

또한 BERT는 문장 앞뒤에 다음과 같은 **특수 토큰(special token)**을 자동으로 붙입니다.

- `[CLS]` — 문장의 맨 앞에 항상 붙는 토큰입니다. 문장 전체를 대표하는 벡터가 필요한 작업(분류 등)에서 사용됩니다. (이 노트북에서는 직접 사용하지 않지만, 항상 따라다니는 토큰이라 알아두면 좋습니다)
- `[SEP]` — 문장의 끝, 혹은 두 문장을 이어붙일 때 그 사이를 구분하는 토큰입니다.
- `[PAD]` — 여러 문장을 한 배치(batch)로 묶을 때 길이를 맞추기 위해 채워 넣는 "의미 없는" 토큰입니다.
- `[MASK]` — 우리가 "이 자리에 어떤 단어가 있었을까?"를 맞히고 싶은, 가려진 자리입니다.
- `[UNK]` — vocab에 전혀 없는, 알 수 없는 토큰일 때 사용됩니다.

이제 직접 코드로 확인해봅시다.


In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

sample_text = "Tokenization splits unbelievable words into smaller pieces."

# tokenize(): 문장을 토큰(문자열) 단위로만 쪼개서 보여줍니다. (아직 숫자로 바꾸기 전)
tokens = tokenizer.tokenize(sample_text)
print("토큰 목록:", tokens)
# 결과에서 '##'으로 시작하는 토큰이 있다면, 그것이 바로 WordPiece가 단어를
# 더 작은 조각으로 쪼갠 부분입니다.

# tokenizer(...)를 직접 호출하면 [CLS]/[SEP]까지 자동으로 붙이고,
# 동시에 숫자(id)로 바꾼 뒤, PyTorch 텐서(return_tensors='pt')로 묶어서 돌려줍니다.
encoded = tokenizer(sample_text, return_tensors='pt')

print("\ninput_ids:     ", encoded['input_ids'])
print("attention_mask:", encoded['attention_mask'])
print("token_type_ids:", encoded['token_type_ids'])

# input_ids      : 각 토큰의 정수 id (실제로 모델에 들어가는 값)
# attention_mask : 1이면 "진짜 토큰이니 봐도 된다", 0이면 "[PAD]라서 무시해라"라는 표시
#                  (지금은 문장이 1개뿐이라 패딩이 없어서 전부 1입니다)
# token_type_ids : 문장이 1개면 전부 0이 되고, 문장 2개를 이어붙이면 두 번째 문장은 1이 됩니다.
#                  (이 노트북에서는 항상 문장 1개만 다루므로 전부 0이 나옵니다)

# 다시 사람이 읽을 수 있는 문장으로 되돌려보기 (decode)
print("\n복원 (특수 토큰 포함):", tokenizer.decode(encoded['input_ids'][0]))
print("복원 (특수 토큰 제외):", tokenizer.decode(encoded['input_ids'][0], skip_special_tokens=True))

print("\n[MASK] 토큰 문자열:", tokenizer.mask_token)
print("[MASK] 토큰의 id  :", tokenizer.mask_token_id)


## 2. 모델 살펴보기: BertForMaskedLM은 무엇일까요?

`BertForMaskedLM`은 크게 두 부분으로 이루어져 있습니다.

1. **BERT 본체 (Transformer Encoder)** — Self-Attention 층을 여러 개 쌓아서, 문장 속 각 토큰이 문장 안의 *다른 모든 토큰*(왼쪽이든 오른쪽이든 상관없이)을 참고해서 자기 자신의 의미를 다시 표현하도록 만드는 부분입니다. `bert-base-uncased`는 이 층이 12개 쌓여 있고, 각 토큰을 768차원의 벡터로 표현합니다.
2. **MLM head (출력층)** — 1번에서 나온 각 토큰의 768차원 벡터를, "vocab에 있는 단어 하나하나가 정답일 점수"로 바꿔주는 작은 출력층입니다. 즉, 토큰 하나당 vocab 크기(`bert-base-uncased`는 약 3만 개)만큼의 점수가 나옵니다.

> **참고:** BERT는 GPT 계열 모델과 달리 "미래 토큰을 보지 못하게 가리는" 구조가 *없습니다*. 문장 안의 모든 토큰은 서로를 양방향으로 모두 참조할 수 있고, 여기서 다루는 `attention_mask`는 그저 `[PAD]`처럼 의미 없는 패딩 토큰을 무시하기 위한 장치일 뿐입니다.

이 모델은 이미 BooksCorpus + 위키백과 같은 매우 큰 텍스트로 MLM 학습이 끝난 **사전학습(pre-trained)** 모델입니다. 즉, 우리는 지금 "이미 똑똑해진" BERT를 그냥 불러와서 쓰는 것입니다.


In [ ]:
model = BertForMaskedLM.from_pretrained('bert-base-uncased')

# eval(): 모델을 "추론 모드"로 전환합니다.
# BERT 내부에는 Dropout처럼, 학습할 때만 무작위로 일부 뉴런을 끄는 층이 있습니다.
# eval() 모드에서는 이런 무작위성이 꺼져서, 같은 입력에 항상 같은 출력이 나오게 됩니다.
# (반대로 학습할 때 쓰는 모드는 model.train() 입니다 - 파인튜닝 섹션에서 다시 사용합니다)
model.eval()

print(model.config)
# config에서 특히 눈여겨볼 값들:
#   hidden_size         : 토큰 하나를 표현하는 벡터의 차원 (보통 768)
#   num_hidden_layers   : Transformer Encoder 층의 개수 (보통 12)
#   num_attention_heads : 한 층 안에서 Self-Attention을 몇 개로 나눠서 보는지 (보통 12)
#   vocab_size          : 이 모델이 알고 있는 토큰의 총 개수

# 참고: 모델을 불러올 때 "Some weights ... were not used" 같은 경고가 뜰 수 있습니다.
# bert-base-uncased는 원래 MLM과 함께 NSP(다음 문장 예측)도 같이 학습되었는데,
# BertForMaskedLM은 NSP 관련 부분을 사용하지 않기 때문에 나오는 정상적인 안내이며,
# 우리가 실습할 MLM 기능에는 전혀 영향이 없습니다.


## 3. [MASK] 자리에 어떤 단어가 올지 예측해보기

이제 실제로 빈칸 채우기를 해볼 차례입니다. 전체 흐름은 다음과 같습니다.

1. `[MASK]`가 포함된 문장을 토큰화한다.
2. 문장에서 `[MASK]` 토큰이 **몇 번째 위치**에 있는지 찾는다.
3. 모델에 문장을 통과시켜서(forward pass), 모든 위치에 대한 **logit**(아직 확률이 아닌, 정답일 가능성을 나타내는 점수)을 얻는다.
4. 그중 `[MASK]` 위치의 logit만 꺼내서, **softmax**를 이용해 "vocab 전체에 대한 확률분포"로 바꾼다. (softmax는 점수가 클수록 더 큰 확률을, 그리고 전체 확률의 합이 항상 1이 되도록 만들어주는 함수입니다)
5. 확률이 가장 높은 top-5 후보를 뽑아서 출력한다.

코드를 한 줄씩 실행하면서 각 단계의 shape(텐서 모양)과 값을 직접 확인해보겠습니다.


In [ ]:
# 마스크된 문장 입력
# (원래 transformer 자리에 무엇이 들어갔는지 BERT가 맞힐 수 있을지 확인해봅시다)
text = "The transformer architecture replaced [MASK] in NLP."
inputs = tokenizer(text, return_tensors='pt')

print("토큰화 결과:", tokenizer.tokenize(text))
print("input_ids   :", inputs['input_ids'])

# [MASK] 위치(인덱스) 찾기
# input_ids[0] (배치의 첫 번째 문장)에서, 값이 mask_token_id와 같은 위치를 모두 찾습니다.
# 지금은 [MASK]가 문장에 1개뿐이므로 결과 텐서에는 숫자 1개만 들어 있습니다.
mask_token_index = torch.where(inputs['input_ids'][0] == tokenizer.mask_token_id)[0]
print("[MASK] 위치 인덱스:", mask_token_index)

# 예측 (순전파)
# torch.no_grad(): 추론만 할 것이므로 기울기(gradient)를 계산하지 않겠다는 표시입니다.
# 학습(backpropagation)이 필요 없을 때 이렇게 꺼주면 메모리를 아끼고 속도도 빨라집니다.
with torch.no_grad():
    outputs = model(**inputs)
logits = outputs.logits

# logits의 shape: (batch_size, sequence_length, vocab_size)
#   - batch_size      : 한 번에 처리한 문장 수 (여기서는 1개)
#   - sequence_length : 토큰 개수 ([CLS], [SEP] 포함)
#   - vocab_size      : BERT가 알고 있는 토큰(서브워드) 전체 개수
# 즉 "문장의 각 위치마다, vocab에 있는 모든 단어 각각에 대한 점수"를 가지고 있는 셈입니다.
print("\nlogits의 shape:", logits.shape)

# Top-5 예측
for position in mask_token_index:
    # 1) [MASK] 위치의 logit만 꺼내기. shape: (vocab_size,)
    mask_logits = logits[0, position]

    # 2) logit -> 확률(probability)로 변환 (softmax)
    probs = torch.softmax(mask_logits, dim=-1)

    # 3) 확률이 가장 높은 5개 후보 뽑기
    top5 = torch.topk(probs, k=5)

    print(f"\n[MASK] 위치 {position.item()}의 Top-5 예측 결과")
    for rank, (score, token_id) in enumerate(zip(top5.values, top5.indices), start=1):
        token = tokenizer.decode([token_id])
        print(f"{rank}. {token.strip():<15s} (확률: {score:.4f})")


## 4. 여러 문장으로 실험하기 위해 함수로 만들기

같은 코드를 매번 새로 치기보다, 문장과 top_k 값만 바꿔서 빠르게 실험할 수 있도록 위 과정을 함수 하나로 묶어두겠습니다. 이렇게 만들어두면, 이어지는 "BERT는 정말 양방향일까?" 실험과, 맨 뒤의 "파인튜닝 전/후 비교" 실험에서도 똑같이 재사용할 수 있습니다.


In [ ]:
def predict_mask(sentence, top_k=5):
    """
    [MASK] 토큰이 포함된 문장을 받아서,
    그 자리에 들어갈 가능성이 높은 단어 top_k개와 확률을 출력합니다.
    (문장에 [MASK]가 여러 개 있어도 모두 처리합니다)
    """
    inputs = tokenizer(sentence, return_tensors='pt')
    mask_token_index = torch.where(inputs['input_ids'][0] == tokenizer.mask_token_id)[0]

    if len(mask_token_index) == 0:
        print(f"'{sentence}' 문장에 [MASK] 토큰이 없습니다. 정확히 [MASK]라고 썼는지 확인해주세요.")
        return

    with torch.no_grad():
        logits = model(**inputs).logits

    print(f"문장: {sentence}")
    for position in mask_token_index:
        probs = torch.softmax(logits[0, position], dim=-1)
        topk = torch.topk(probs, k=top_k)
        for rank, (score, token_id) in enumerate(zip(topk.values, topk.indices), start=1):
            token = tokenizer.decode([token_id])
            print(f"  {rank}. {token.strip():<15s} (확률: {score:.4f})")
    print()


# 앞에서 했던 예측을 함수로 다시 해보기 (결과가 같은지 확인)
predict_mask("The transformer architecture replaced [MASK] in NLP.")


## 5. BERT는 정말 "양방향(Bidirectional)"일까요? 직접 확인해보기

BERT 논문의 핵심 주장은 "MLM으로 학습하면, 모델이 `[MASK]`의 **왼쪽**뿐 아니라 **오른쪽** 문맥까지 한꺼번에 활용해서 단어를 예측하게 된다"는 것입니다. 말로만 들으면 추상적이니, 직접 문장을 바꿔보면서 눈으로 확인해봅시다.

먼저 아래 문장을 보세요.

> `The [MASK] barked loudly at the mailman.`

여기서 `[MASK]` **왼쪽**만 보면 "The ___"인데, 이 정보만으로는 빈칸에 어떤 단어든 들어갈 수 있어서 거의 아무런 단서가 없습니다. 하지만 `[MASK]` **오른쪽**에 있는 "barked loudly at the mailman"이라는 표현 덕분에, 우리는 (그리고 BERT도) "짖는(barked) 주체이니 동물, 그중에서도 개일 가능성이 높겠다"라고 답을 좁혀나갈 수 있습니다.

즉, **오른쪽 문맥이 없으면 거의 예측이 불가능한 문장**을 BERT에게 줘보고, 정말 그럴듯한 답(`dog`, `puppy` 등 동물 관련 단어)을 내놓는지 확인하면, BERT가 실제로 오른쪽 문맥까지 같이 활용하고 있다는 것을 간접적으로 확인할 수 있습니다.

비교를 위해, 오른쪽 문맥을 거의 없애버린 문장도 같이 실행해서 예측이 얼마나 더 막연해지는지(= 확률이 낮고 답이 두루뭉술해지는지) 비교해보겠습니다.


In [ ]:
# (A) 오른쪽 문맥이 풍부한 경우
predict_mask("The [MASK] barked loudly at the mailman.")

# (B) 오른쪽 문맥을 거의 없애버린 경우 (왼쭉 정보만 있는 것과 비슷한 상황)
predict_mask("The [MASK] was there.")

# (A)와 (B)의 top-5 결과를 비교해보세요.
# (A)에서는 dog, cat, puppy 처럼 "짖을 수 있는 대상"으로 답이 좁혀지는 경향이 있고,
# (B)에서는 문맥 정보가 부족해서 답이 훨씬 다양하고 확률(점수)도 더 낮게 퍼지는 경향이 있을 것입니다.
# (실제로 어떤 단어가 1위로 나오는지는 직접 실행해서 확인해보세요!)


같은 원리로, 문맥에 따라 전혀 다른 단어가 정답이 되는 경우도 살펴봅시다. 아래 두 문장은 `[MASK]` 자리의 "역할"은 비슷해 보이지만(둘 다 명사 자리), 주변 단어가 완전히 다르기 때문에 전혀 다른 답이 나오는 것을 볼 수 있습니다.


In [ ]:
predict_mask("I went to the [MASK] to deposit my paycheck.")
predict_mask("She poured the batter into the [MASK] before baking it.")


## 6. 이제 직접 파인튜닝(Fine-tuning)을 해봅시다

지금까지 사용한 `bert-base-uncased`는 이미 매우 큰 범용 텍스트(BooksCorpus + 위키백과)로 MLM을 학습한 **사전학습(pre-training)** 모델입니다. 그런데 만약 우리가 다루는 텍스트가 어떤 특정 분야(예: 의학, 법률, 특정 서비스의 사용자 리뷰 등)에 좀 더 특화되어 있다면 어떨까요? 범용 모델은 그 분야의 용어/표현에 최적화되어 있지 않을 수 있습니다.

이럴 때 사용하는 방법이 **파인튜닝(fine-tuning)**입니다. 이미 학습이 끝난 모델의 가중치를 가져와서, 더 작고 특정 분야에 맞는 데이터로 "조금 더" 학습시키는 것입니다. (가중치를 0부터 다시 학습시키는 사전학습과 달리, 이미 똑똑한 상태에서 시작하기 때문에 적은 데이터/적은 시간으로도 효과를 볼 수 있습니다.)

여기서는 **'커피숍'을 주제로 한 짧은 영어 문장 모음**을 직접 만들어서, BERT의 MLM 기능을 그 도메인에 맞게 한 번 더 학습시켜보겠습니다.

### BERT 논문의 마스킹 규칙 (15%, 그리고 80/10/10)

BERT 논문에서는 학습 데이터를 만들 때 다음 규칙으로 마스킹합니다.

1. 문장의 토큰 중 **15%**를 무작위로 선택합니다.
2. 선택된 토큰에 대해서:
   - **80%**의 경우 → `[MASK]`로 바꿉니다.
   - **10%**의 경우 → 전혀 다른 무작위 토큰으로 바꿔버립니다.
   - **10%**의 경우 → 원래 토큰을 그대로 둡니다.
3. 위 세 가지 경우 모두, 모델은 "원래 그 자리에 있던 정답 토큰"을 맞히도록 학습됩니다.

> **왜 이렇게 복잡하게 할까요?** 만약 항상 `[MASK]`로만 바꾼다면, 모델은 "`[MASK]`가 보일 때만" 열심히 추론하는 버릇이 생길 수 있습니다. 그런데 실제 파인튜닝/실제 사용 시점에는 `[MASK]` 토큰이 전혀 등장하지 않습니다! 그래서 가끔은 무작위 토큰으로 바꾸거나 아예 그대로 두어서, 모델이 "모든 토큰"에 대해 항상 좋은 표현(representation)을 만들도록 유도하는 것입니다.

다행히 이 규칙을 직접 코드로 짤 필요는 없습니다. HuggingFace의 `DataCollatorForLanguageModeling`이 이 과정을 자동으로 처리해줍니다. 바로 아래에서 어떻게 동작하는지 직접 눈으로 확인해보겠습니다.


In [ ]:
# '커피숍/카페'를 주제로 한 작은 말뭉치(corpus)를 직접 만들어봅니다.
# 실제 연구/실무에서는 보통 수천~수백만 문장 단위의 데이터를 사용하지만,
# 여기서는 "파인튜닝으로 모델의 예측이 실제로 바뀌는 모습"을 직접 눈으로
# 확인하는 것이 목적이므로, 의도적으로 아주 작고 주제가 뚜렷한 데이터를 사용합니다.
cafe_corpus = [
    "I ordered a cappuccino and a croissant at the cafe this morning.",
    "The barista recommended a single origin espresso from Ethiopia.",
    "She always adds oat milk to her latte instead of regular milk.",
    "We sat by the window and shared a slice of chocolate cake.",
    "The coffee shop on the corner roasts its own beans every week.",
    "He likes his coffee black, with no sugar or cream at all.",
    "The cafe was crowded, so we waited fifteen minutes for a table.",
    "A warm cup of coffee is the best way to start a cold morning.",
    "The barista drew a small heart in the foam of my latte.",
    "Could I get a refill on my drip coffee, please?",
    "The menu offers cold brew, iced latte, and hot americano.",
    "I love the smell of freshly ground coffee beans in the morning.",
    "The cafe plays soft jazz music while customers read or chat.",
    "She ordered a decaf coffee because it was already late at night.",
    "The pastry case was full of muffins, scones, and cinnamon rolls.",
    "Our favorite cafe has cozy chairs and a fireplace in the corner.",
    "He spilled his coffee on the table and quickly asked for a napkin.",
    "The new cafe near the station serves the best flat white in town.",
    "I usually study at the cafe because the wifi is fast and free.",
    "The barista steamed the milk until it was smooth and silky.",
]

print(f"문장 개수: {len(cafe_corpus)}")
print("예시 문장 3개:")
for sentence in cafe_corpus[:3]:
    print(" -", sentence)


파인튜닝의 효과를 "전/후"로 비교하려면, 학습을 시작하기 *전*의 예측 결과를 먼저 기록해두어야 합니다. 아래 두 문장으로 지금 시점의 예측을 확인해두세요. 파인튜닝이 끝난 뒤, 똑같은 두 문장으로 다시 예측해서 결과가 어떻게 달라지는지 비교할 것입니다.


In [ ]:
print("=== 파인튜닝 전 예측 결과 ===\n")
predict_mask("I ordered a [MASK] with extra foam at the coffee shop.")
predict_mask("The barista steamed the [MASK] until it was smooth and silky.")


### 데이터를 모델이 학습할 수 있는 형태로 바꾸기

문장 리스트(Python list)를 그대로 모델에 넣을 수는 없습니다. 먼저 HuggingFace `datasets` 라이브러리의 `Dataset` 형태로 바꾸고, 그 안의 모든 문장을 tokenizer로 토큰화해야 합니다.


In [ ]:
from datasets import Dataset

# 문장 리스트를 {"text": [...]} 형태의 딕셔너리로 감싸서 Dataset으로 변환합니다.
# 이렇게 하면 "text"라는 이름의 열(column)이 하나 있는 표(table) 같은 구조가 됩니다.
raw_dataset = Dataset.from_dict({"text": cafe_corpus})
print(raw_dataset)


def tokenize_function(examples):
    # truncation=True : 문장이 모델의 최대 입력 길이(보통 512)를 넘으면 잘라냅니다.
    #                    지금은 모두 짧은 문장이라 실제로 잘릴 일은 거의 없지만,
    #                    혹시 모를 긴 문장에 대비한 안전장치로 넣어둡니다.
    return tokenizer(examples["text"], truncation=True)


# .map(..., batched=True): 문장을 한 개씩이 아니라 여러 개를 한꺼번에 토큰화해서 더 빠르게 처리합니다.
# remove_columns=["text"]: 토큰화가 끝나면 더 이상 필요 없는 원본 문자열 열은 제거합니다.
#                          (모델은 문자열을 직접 입력으로 받을 수 없기 때문입니다)
tokenized_dataset = raw_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

print("\n토큰화 후 첫 번째 데이터:")
print(tokenized_dataset[0])


### 마스킹을 자동으로 해주는 DataCollatorForLanguageModeling

앞서 설명한 "15%를 고르고, 그중 80/10/10 비율로 처리"하는 규칙을 매번 직접 구현하는 대신, `DataCollatorForLanguageModeling`을 사용하면 배치(batch)를 만들 때마다 자동으로 이 마스킹을 적용해줍니다. (그리고 같은 문장이라도 epoch마다 가려지는 위치가 달라집니다 - 이 역시 논문의 방식과 동일합니다.)

또한 이 collator는 길이가 서로 다른 문장들을 하나의 배치로 묶을 때 필요한 **패딩(padding)**까지 자동으로 처리해줍니다.

직접 작은 데이터로 어떻게 동작하는지 확인해봅시다.


In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,              # Masked LM 방식의 마스킹을 사용한다는 의미입니다 (기본값 True)
    mlm_probability=0.15,  # 논문과 동일하게 토큰의 15%를 마스킹 대상으로 고릅니다.
)

# data_collator가 실제로 어떤 식으로 마스킹을 적용하는지, 문장 2개만 가지고 직접 확인해보기
sample_batch = [tokenized_dataset[i] for i in range(2)]
collated = data_collator(sample_batch)

print("마스킹 + 패딩이 적용된 input_ids:\n", collated['input_ids'])
print("\n정답 labels:\n", collated['labels'])

# labels를 해석하는 방법:
#   -100      : "이 위치는 손실(loss) 계산에서 제외하라"는 의미입니다.
#               즉, 마스킹되지 않은(원래 그대로 보이는) 토큰의 자리는 모두 -100으로 채워집니다.
#   그 외 숫자 : 마스킹된 위치의 "원래 정답 토큰 id"입니다.
#               모델은 바로 이 위치들에 대해서만 "원래 뭐였는지"를 맞히도록 학습됩니다.
print("\n실제로 마스킹된(= label이 -100이 아닌) 토큰 개수:",
      (collated['labels'] != -100).sum().item())


### loss는 어떻게 계산될까요?

`BertForMaskedLM`은 입력으로 `labels`까지 함께 받으면, 내부적으로 다음과 같은 작업을 자동으로 수행하고 그 결과를 `outputs.loss`로 돌려줍니다.

1. 모든 위치에 대해 logits을 계산합니다. (shape: `(batch, seq_len, vocab_size)`)
2. 각 위치의 logits과, 그 위치의 label(정답 토큰 id, 또는 -100)을 비교해서 **Cross Entropy Loss**를 계산합니다.
3. label이 `-100`인 위치는 계산에서 제외합니다. (즉 마스킹된 위치들만 손실 계산에 반영됩니다)

말로만 들으면 "정말 그렇게 동작하는지" 궁금할 수 있으니, 우리가 직접 동일한 계산을 `torch.nn.functional.cross_entropy`로 한 번 더 해보고, `outputs.loss`와 값이 똑같이 나오는지 검증해보겠습니다.


In [ ]:
import torch.nn.functional as F

with torch.no_grad():
    outputs = model(**collated)  # labels가 포함된 배치를 그대로 넣어줍니다.

logits = outputs.logits
model_loss = outputs.loss

# Cross entropy는 입력으로 (전체 토큰 개수, vocab_size) 모양의 logits과
# (전체 토큰 개수,) 모양의 labels를 기대합니다.
# 따라서 (batch, seq_len, vocab_size) -> (batch*seq_len, vocab_size) 로 펼쳐줍니다.
manual_loss = F.cross_entropy(
    logits.view(-1, logits.size(-1)),
    collated['labels'].view(-1),
    ignore_index=-100,  # label이 -100인 위치는 계산에서 제외
)

print("BertForMaskedLM이 직접 계산한 loss :", model_loss.item())
print("우리가 수동으로 계산한 loss        :", manual_loss.item())
print("두 값이 거의 같은가?               :", torch.allclose(model_loss, manual_loss, atol=1e-4))


### 학습 루프 (Training Loop) 직접 작성하기

이제 실제로 모델을 학습시켜 보겠습니다. 한 번의 학습 스텝(step)은 보통 다음 네 단계로 이루어집니다.

1. **순전파(forward)** — 입력을 모델에 넣어 logits과 loss를 구합니다.
2. **역전파(backward)** — `loss.backward()`를 호출해서, loss를 줄이려면 각 파라미터를 어느 방향으로 움직여야 하는지(기울기, gradient)를 계산합니다.
3. **파라미터 업데이트** — `optimizer.step()`으로, 계산된 기울기 방향으로 모델의 파라미터를 아주 조금씩 이동시킵니다.
4. **기울기 초기화** — `optimizer.zero_grad()`로, 다음 배치를 위해 누적된 기울기를 0으로 리셋합니다. (이걸 안 하면 기울기가 계속 쌓여서 잘못된 방향으로 학습됩니다)

이 네 단계를 데이터셋 전체에 대해 한 바퀴 도는 것을 **1 epoch**이라고 부릅니다. 데이터가 워낙 작으므로, 여러 epoch을 반복해서 학습시키겠습니다.


In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

torch.manual_seed(42)  # 마스킹/셔플의 무작위성을 고정해서, 다시 실행해도 비슷한 흐름을 보기 위함입니다.

# DataLoader: tokenized_dataset에서 미니배치(mini-batch)를 하나씩 꺼내주는 역할을 합니다.
# collate_fn으로 위에서 만든 data_collator를 지정하면, 배치를 꺼낼 때마다
# "그 순간" 새롭게 무작위 마스킹을 적용해줍니다 (= 매 epoch마다 가려지는 위치가 달라짐).
train_dataloader = DataLoader(
    tokenized_dataset,
    batch_size=4,
    shuffle=True,           # 매 epoch마다 문장 순서를 섞어서, 순서에 의한 편향을 줄입니다.
    collate_fn=data_collator,
)

# AdamW: BERT 계열 모델을 파인튜닝할 때 가장 널리 쓰이는 옵티마이저입니다.
# lr(learning rate)=5e-5는 BERT 논문에서도 추천하는 대표적인 파인튜닝 학습률 값입니다.
optimizer = AdamW(model.parameters(), lr=5e-5)

model.train()  # 학습 모드로 전환! (eval()의 반대 - Dropout이 다시 활성화됩니다)

num_epochs = 8
for epoch in range(num_epochs):
    total_loss = 0.0
    for batch in train_dataloader:
        outputs = model(**batch)   # labels가 포함되어 있으므로 loss까지 한 번에 계산됩니다.
        loss = outputs.loss

        loss.backward()           # 역전파: 기울기 계산
        optimizer.step()          # 파라미터 한 걸음 업데이트
        optimizer.zero_grad()     # 다음 배치를 위해 기울기 초기화

        total_loss += loss.item()

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{num_epochs} - 평균 loss: {avg_loss:.4f}")

# 이 작은 데이터셋(20문장) + 적은 epoch 기준으로는, CPU에서도 보통
# 수십 초 ~ 몇 분 이내에 끝납니다. (GPU를 쓰면 훨씬 더 빠릅니다)


## 7. 파인튜닝 전/후 비교해보기

학습이 끝났습니다. 이제 model을 다시 추론 모드로 돌리고, 맨 처음 기록해두었던 것과 **똑같은 두 문장**으로 다시 예측해봅시다. 아래 항목들을 눈으로 비교해보세요.

- top-1 후보 단어가 카페/커피와 더 관련 있는 단어로 바뀌었나요?
- 카페 관련 단어(latte, milk, coffee 등)가 top-5 안에 새로 등장했나요?
- 가장 높은 확률(점수) 값 자체가 더 올라갔나요? (모델이 더 "확신"하게 되었다는 뜻입니다)

> **참고:** 우리가 사용한 데이터는 20문장뿐이고 학습도 매우 짧게 진행했기 때문에, 실제 프로덕션 수준의 파인튜닝만큼 극적인 변화가 보이지 않을 수도 있습니다. 이 실습의 목적은 "파인튜닝이 모델의 예측 분포를 실제로 바꾼다"는 메커니즘을 직접 체감하는 것이며, 실제 프로젝트에서는 훨씬 더 많은 데이터와 학습이 필요합니다.


In [ ]:
model.eval()  # 다시 추론 모드로 전환

print("=== 파인튜닝 후 예측 결과 ===\n")
predict_mask("I ordered a [MASK] with extra foam at the coffee shop.")
predict_mask("The barista steamed the [MASK] until it was smooth and silky.")

print("위 결과를 파인튜닝 '전' 결과(위쪽 셀의 출력)와 비교해보세요!")


## 마무리 정리

이 노트북에서 직접 다뤄본 내용을 정리하면 다음과 같습니다.

- **Tokenizer**: 문장을 WordPiece 토큰으로 자르고, `[CLS]`/`[SEP]`/`[MASK]`/`[PAD]` 같은 특수 토큰이 각각 어떤 역할을 하는지 확인했습니다.
- **MLM 추론**: `logits -> softmax -> top-k` 흐름으로, 사전학습된 BERT가 `[MASK]` 자리의 단어를 어떻게 예측하는지 한 줄씩 따라가 보았습니다.
- **양방향성**: `[MASK]` 오른쪽 문맥이 있을 때와 없을 때 예측이 얼마나 달라지는지 비교하면서, BERT가 왜 "양방향" 모델인지 직접 체감했습니다.
- **파인튜닝**: BERT 논문의 마스킹 규칙(15%, 80/10/10)이 `DataCollatorForLanguageModeling`으로 어떻게 자동 구현되는지 확인하고, `-100` label과 loss 계산 방식을 직접 검증한 뒤, 작은 커피숍 말뭉치로 실제로 모델을 한 번 더 학습시켜 보았습니다.

### 다음으로 시도해볼 만한 것들

- `cafe_corpus`를 직접 원하는 주제(예: 영화 리뷰, 요리 레시피 등)로 바꿔서 같은 실험을 반복해보세요.
- `mlm_probability`, `num_epochs`, learning rate 같은 값을 바꿔보면서 결과가 어떻게 달라지는지 실험해보세요.
- BERT는 MLM 외에도 분류(classification), 질의응답(QA), 개체명 인식(NER) 같은 다양한 다운스트림 태스크에 파인튜닝할 수 있습니다. 이때는 `BertForMaskedLM` 대신 `BertForSequenceClassification`, `BertForQuestionAnswering` 같은 다른 head를 가진 클래스를 사용합니다.
